In [ ]:
# ΔOD with time-resolved blank subtraction — MinMedia only (all replicates)
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib

file_path = './MicroCohnella_MinRichMedia_0924.xlsx'
df = pd.read_excel(file_path, sheet_name='Sheet1')

# Keep rows with time, cap to 150 h, sort
time_col = 'Time (hour)'
df = df[~df[time_col].isna()].copy()
df = df[df[time_col] <= 150].sort_values(time_col).reset_index(drop=True)
time = df[time_col].to_numpy()

# -------- column discovery: MinMedia only (all replicates) --------
species_list = ["Cohnella", "Microbacterium"]  # COHM and MIP
media_only = "MinMedia"

def cols(kind, media, species):
    # Matches headers like "MinMedia_inoc_Cohnella_*" (adjust if your order differs)
    needle = f"{media}_{kind}_{species}"
    return [c for c in df.columns if needle in str(c)]

inoculated_conditions, blank_conditions = {}, {}
for sp in species_list:
    key = f"{sp}_{media_only}"
    inoculated_conditions[key] = cols("inoc",  media_only, sp)   # ALL replicates
    blank_conditions[key]      = cols("blank", media_only, sp)   # ALL blanks

# Prefer exact t0 if present, else earliest
t0_idx = (df.index[df[time_col] == 0].tolist() or [0])[0]

plot_data = {}
blank_usage_records = []

for key, inoc_cols in inoculated_conditions.items():
    if not inoc_cols:
        continue

    # Mean blank per timepoint (0 if none)
    blank_cols = blank_conditions.get(key, [])
    if blank_cols:
        blank_numeric = df[blank_cols].apply(pd.to_numeric, errors='coerce')
        blank_mean = blank_numeric.mean(axis=1, skipna=True)
    else:
        blank_mean = pd.Series(0.0, index=df.index)

    for bc in (blank_cols or ["<none>"]):
        blank_usage_records.append({"Condition": key, "BlankColumn": bc})

    # Blank-correct each replicate
    inoc_numeric = df[inoc_cols].apply(pd.to_numeric, errors='coerce')
    corrected = inoc_numeric.subtract(blank_mean, axis='index')

    # Baseline per replicate (t0 if available, else first non-NaN)
    baselines = {}
    for col in corrected.columns:
        if t0_idx in corrected.index and pd.notna(corrected.loc[t0_idx, col]):
            base_val = float(corrected.loc[t0_idx, col])
        else:
            non_nan = corrected[col].dropna()
            base_val = float(non_nan.iloc[0]) if not non_nan.empty else 0.0
        baselines[col] = base_val

    delta = corrected.subtract(pd.Series(baselines), axis='columns')

    # Time-resolved mean ± SE across ALL replicates
    mean = delta.mean(axis=1, skipna=True)
    std  = delta.std(axis=1, ddof=1, skipna=True)
    n    = delta.count(axis=1)
    se   = (std / np.sqrt(n.replace(0, np.nan))).fillna(0.0)

    plot_data[key] = {"mean": mean.values, "se": se.values}

# ---------------- plot (same formatting; solid lines; color map MIP=blue, COHM=red) ----------------
fig, ax = plt.subplots(figsize=(10, 6))

color_map = {"Cohnella": "red", "Microbacterium": "blue"}  # COHM->red, MIP->blue
for key, data in plot_data.items():
    mean, se = data["mean"], data["se"]
    sp = "Cohnella" if key.startswith("Cohnella") else "Microbacterium"
    color = color_map[sp]
    linestyle = "-"  # solid for both

    (ln,) = ax.plot(time, mean, label=key, color=color, linestyle=linestyle, linewidth=2)
    ln.set_clip_on(False); ln.set_clip_path(None)

    fb = ax.fill_between(time, mean - se, mean + se, alpha=0.2, color=color)
    fb.set_clip_on(False); fb.set_clip_path(None)

# X stops at 150; let Y auto; don't force origin
ax.set_xlim(0, 150)

ax.set_title("ΔOD600 vs t=0 — MinMedia only (all replicates)")
ax.set_xlabel("Time (hour)")
ax.set_ylabel("Mean ΔOD600")
ax.legend(title="Condition", bbox_to_anchor=(1.05, 1), loc='upper left')
ax.grid(True, linewidth=0.6, alpha=0.4)
fig.tight_layout()

pdf_out = "./growth_MinMedia_all_reps_COHMred_MIPblue.pdf"
fig.savefig(pdf_out, dpi=300, bbox_inches="tight")

# Optional: blanks used
blank_usage_df = pd.DataFrame(blank_usage_records).drop_duplicates().reset_index(drop=True)
print(pdf_out)
print(blank_usage_df)


In [ ]:
# Compute ΔOD with time-resolved blank subtraction (same formatting) and show how blanks were used
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib

file_path = './MicroCohnella_MinRichMedia_0924.xlsx'
df = pd.read_excel(file_path, sheet_name='Sheet1')

# Keep rows with time, cap to 150 h, and sort
time_col = 'Time (hour)'
df = df[~df[time_col].isna()].copy()
df = df[df[time_col] <= 150].sort_values(time_col).reset_index(drop=True)
time = df[time_col].to_numpy()

# Identify inoculated and blank columns by species + media
species_list = ["Cohnella", "Microbacterium"]
media_list = ["MinMedia", "RichMedia"]

def cols(kind, media, species):
    # kind: "inoc" or "blank" — matches headers like "MinMedia_inoc_Cohnella"
    needle = f"{media}_{kind}_{species}"
    return [c for c in df.columns if needle in str(c)]

# Build condition dicts (same labels as your plot)
inoculated_conditions = {}
blank_conditions = {}
for sp in species_list:
    for med in media_list:
        key = f"{sp}_{med}"
        inoculated_conditions[key] = cols("inoc", med, sp)
        blank_conditions[key] = cols("blank", med, sp)

# Prefer exact t0 if present
t0_rows = df.index[df[time_col] == 0].tolist()
t0_idx = t0_rows[0] if t0_rows else 0

plot_data = {}
blank_usage_records = []

for key, inoc_cols in inoculated_conditions.items():
    if not inoc_cols:
        continue

    # Corresponding blanks for this species/media
    blank_cols = blank_conditions.get(key, [])

    # Compute mean blank per timepoint (if no blanks, it's 0)
    if blank_cols:
        blank_numeric = df[blank_cols].apply(pd.to_numeric, errors='coerce')
        blank_mean = blank_numeric.mean(axis=1, skipna=True)
    else:
        blank_mean = pd.Series(0.0, index=df.index)

    # Record which blank columns were used
    for bc in (blank_cols or ["<none>"]):
        blank_usage_records.append({"Condition": key, "BlankColumn": bc})

    # Blank-correct each inoculated replicate, then compute Δ vs its baseline
    inoc_numeric = df[inoc_cols].apply(pd.to_numeric, errors='coerce')
    corrected = inoc_numeric.subtract(blank_mean, axis='index')

    # Baseline per replicate column: value at t0 if available, else first non-NaN
    baselines = {}
    for col in corrected.columns:
        if t0_idx in corrected.index and pd.notna(corrected.loc[t0_idx, col]):
            base_val = float(corrected.loc[t0_idx, col])
        else:
            non_nan = corrected[col].dropna()
            base_val = float(non_nan.iloc[0]) if not non_nan.empty else 0.0
        baselines[col] = base_val

    baseline_series = pd.Series(baselines)
    delta = corrected.subtract(baseline_series, axis='columns')

    # Mean ± SE across replicates at each timepoint
    mean = delta.mean(axis=1, skipna=True)
    std  = delta.std(axis=1, ddof=1, skipna=True)
    n    = delta.count(axis=1)
    se   = (std / np.sqrt(n.replace(0, np.nan))).fillna(0.0)

    plot_data[key] = {"mean": mean.values, "se": se.values}

# ---- Plot with same formatting (line + shaded SE area) ----
fig, ax = plt.subplots(figsize=(10, 6))

for key, data in plot_data.items():
    mean, se = data["mean"], data["se"]
    color = "blue" if "MinMedia" in key else "red"
    linestyle = "-" if key.startswith("Cohnella") else "--"

    # main line
    (ln,) = ax.plot(time, mean, label=key, color=color, linestyle=linestyle, linewidth=2)
    ln.set_clip_on(False)
    ln.set_clip_path(None)

    # SE ribbon (area)
    fb = ax.fill_between(time, mean - se, mean + se, alpha=0.2, color=color)
    fb.set_clip_on(False)
    fb.set_clip_path(None)

# Axes: start both at zero and stop x at 150
ax.set_xlim(0, 150)
ymax = max(np.nanmax(np.asarray(v["mean"]) + np.asarray(v["se"])) for v in plot_data.values())
ax.set_ylim(0, ymax * 1.08)  # a touch of headroom

# Remove default margins so the axes meet at the origin
ax.margins(x=0, y=0)

ax.set_title("Combined Growth Curves (ΔOD600 vs t=0) with Blank Subtraction")
ax.set_xlabel("Time (hour)")
ax.set_ylabel("Mean ΔOD600")
ax.legend(title="Condition", bbox_to_anchor=(1.05, 1), loc='upper left')
ax.grid(True, linewidth=0.6, alpha=0.4)
fig.tight_layout()

pdf_out = "./combined_growth_curves_deltaOD_blank_corrected_origin0.pdf"
fig.savefig(pdf_out, dpi=300, bbox_inches="tight")

# Report which blank columns were used
blank_usage_df = pd.DataFrame(blank_usage_records).drop_duplicates().reset_index(drop=True)
print(pdf_out)
print(blank_usage_df)


In [ ]:
# ============================================================
# ΔOD600 growth curves:
#   - Microbacterium MinMedia (time-series, blank-subtracted)
#   - Microbacterium Succinate (simulated from growth parameters y0, mumax, K)
# Outliers removed for Microbacterium succinate parameters (IQR filter)
# ============================================================

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
matplotlib.use("cairo")  # for headless environments
# ------------------------------------------------------------
# FILES
# ------------------------------------------------------------
minmedia_file = "./MicroCohnella_MinRichMedia_0924.xlsx"
succinate_params_file = "./SynCom_growthparameters_succinate.xlsx"

# ------------------------------------------------------------
# LOAD MINMEDIA (YOUR ORIGINAL PIPELINE, Microbacterium only)
# ------------------------------------------------------------
df_min = pd.read_excel(minmedia_file, sheet_name="Sheet1")

time_col = "Time (hour)"
df_min = df_min[~df_min[time_col].isna()].copy()
df_min = df_min[df_min[time_col] <= 150].sort_values(time_col).reset_index(drop=True)
time = df_min[time_col].to_numpy()

def cols(df, kind, media, species):
    needle = f"{media}_{kind}_{species}"
    return [c for c in df.columns if needle in str(c)]

# Microbacterium MinMedia only
inoc_cols = cols(df_min, "inoc", "MinMedia", "Microbacterium")
blank_cols = cols(df_min, "blank", "MinMedia", "Microbacterium")

if not inoc_cols:
    raise ValueError("No MinMedia inoculated Microbacterium columns found. Check column headers.")

blank_mean = (
    df_min[blank_cols].apply(pd.to_numeric, errors="coerce").mean(axis=1, skipna=True)
    if blank_cols else pd.Series(0.0, index=df_min.index)
)

inoc = df_min[inoc_cols].apply(pd.to_numeric, errors="coerce")
corrected = inoc.subtract(blank_mean, axis="index")

# baseline at t0
t0_idx = (df_min.index[df_min[time_col] == 0].tolist() or [0])[0]
baselines = {}
for c in corrected.columns:
    if pd.notna(corrected.loc[t0_idx, c]):
        baselines[c] = float(corrected.loc[t0_idx, c])
    else:
        non_nan = corrected[c].dropna()
        baselines[c] = float(non_nan.iloc[0]) if not non_nan.empty else 0.0

delta_min = corrected.subtract(pd.Series(baselines), axis="columns")

mean_min = delta_min.mean(axis=1, skipna=True)
std_min  = delta_min.std(axis=1, ddof=1, skipna=True)
n_min    = delta_min.count(axis=1)
se_min   = (std_min / np.sqrt(n_min.replace(0, np.nan))).fillna(0.0)

# ------------------------------------------------------------
# LOAD SUCCINATE GROWTH PARAMETERS (NOT TIME SERIES)
# Columns: strain_name, y0, mumax, K (and maybe others)
# ------------------------------------------------------------
df_par = pd.read_excel(succinate_params_file)

needed = {"strain_name", "y0", "mumax", "K"}
missing = needed - set(df_par.columns)
if missing:
    raise ValueError(f"Succinate parameters file is missing columns: {missing}. "
                     f"Found columns: {list(df_par.columns)}")

mb = df_par[df_par["strain_name"].astype(str).str.strip().eq("Microbacterium")].copy()
if mb.empty:
    raise ValueError("No rows found for strain_name == 'Microbacterium' in succinate parameter file.")

# Ensure numeric
for c in ["y0", "mumax", "K"]:
    mb[c] = pd.to_numeric(mb[c], errors="coerce")
mb = mb.dropna(subset=["y0", "mumax", "K"]).reset_index(drop=True)

# ------------------------------------------------------------
# OUTLIER REMOVAL (IQR) ACROSS REPLICATES, for each parameter
# Remove replicate if it is an outlier in ANY of y0, mumax, or K
# ------------------------------------------------------------
def iqr_bounds(s, k=1.5):
    q1 = s.quantile(0.25)
    q3 = s.quantile(0.75)
    iqr = q3 - q1
    lo = q1 - k * iqr
    hi = q3 + k * iqr
    return lo, hi

mask_keep = pd.Series(True, index=mb.index)
for c in ["y0", "mumax", "K"]:
    lo, hi = iqr_bounds(mb[c])
    mask_keep &= mb[c].between(lo, hi)

mb_clean = mb[mask_keep].copy().reset_index(drop=True)

print(f"Microbacterium succinate replicates: {len(mb)} total")
print(f"Kept after outlier removal: {len(mb_clean)}")
print("Removed replicate rows (outliers):")
print(mb.loc[~mask_keep, ["strain_no", "strain_name", "y0", "mumax", "K"]] if "strain_no" in mb.columns else mb.loc[~mask_keep, ["strain_name", "y0", "mumax", "K"]])

if len(mb_clean) < 2:
    print("WARNING: <2 replicates kept after outlier removal. SE band may be unreliable.")

# ------------------------------------------------------------
# SIMULATE LOGISTIC GROWTH CURVES FROM PARAMETERS
# Logistic: OD(t) = K / (1 + ((K - y0)/y0)*exp(-mumax*t))
# Then ΔOD = OD(t) - OD(t0)
# ------------------------------------------------------------
def logistic(t, y0, mumax, K):
    # guard against division issues
    y0 = max(float(y0), 1e-9)
    K  = max(float(K),  1e-9)
    return K / (1.0 + ((K - y0) / y0) * np.exp(-float(mumax) * t))

# simulate per replicate, then mean ± SE
curves = []
for _, row in mb_clean.iterrows():
    od = logistic(time, row["y0"], row["mumax"], row["K"])
    delta = od - od[0]  # t0 is time[0] (should be 0 in your MinMedia file)
    curves.append(delta)

curves = np.vstack(curves)  # shape: (n_reps, n_time)
mean_suc = np.nanmean(curves, axis=0)
std_suc  = np.nanstd(curves, axis=0, ddof=1) if curves.shape[0] > 1 else np.zeros_like(mean_suc)
se_suc   = std_suc / np.sqrt(curves.shape[0]) if curves.shape[0] > 0 else np.zeros_like(mean_suc)

# ------------------------------------------------------------
# PLOT
# ------------------------------------------------------------
fig, ax = plt.subplots(figsize=(10, 6))

# Microbacterium MinMedia (blue)
ax.plot(time, mean_min.values, color="blue", lw=2, label="Microbacterium – MinMedia")
ax.fill_between(time, (mean_min - se_min).values, (mean_min + se_min).values,
                color="blue", alpha=0.2)

# Microbacterium Succinate (red) -- simulated from parameters
ax.plot(time, mean_suc, color="red", lw=2, label="Microbacterium – Succinate (simulated)")
ax.fill_between(time, mean_suc - se_suc, mean_suc + se_suc,
                color="red", alpha=0.2)

ax.set_xlim(0, 150)
ax.set_xlabel("Time (hour)")
ax.set_ylabel("Mean ΔOD600")
ax.set_title("Microbacterium growth: MinMedia (measured) vs Succinate (simulated)")
ax.legend()
ax.grid(True, alpha=0.4)

fig.tight_layout()
out = "Microbacterium_MinMedia_vs_Succinate.pdf"
fig.savefig(out, dpi=300, bbox_inches="tight")
print("Saved:", out)
